# Lab: GenAI Cost Management

📚 In this lab you will learn and practice the following:

❄️ How to review the Cortex metering usage

❄️ Review cost management Dashboard

❄️ Understand cost management guidelines

❄️ Set up resource budgets for Cortex Agents and Snowflake CoWork

📌 **Note**:

Due to **regional workload spikes**, there may be **latency** with some of the steps. In the real world, for consistent performance, customers can explore [**Provisioned Throughput**](https://docs.snowflake.com/en/user-guide/snowflake-cortex/provisioned-throughput).

If you find your queries are running for more than 5 minutes, cancel and come back and try them later.

If you find models that are deprecated, use CoCo to help you fix the issue by selecting a suitable model.

---

### 🤖 Use CoCo as you go!

> **💡 TIP 1**: Use CoCo to explain complex SQL statements. Select any query and ask *"Explain this SQL"* to get a plain-language breakdown of what it does.
>
> **💡 TIP 2**: Want to learn more about any function? Ask CoCo *"What does [function name] do?"* to get its syntax, supported options, and examples.
>
> **💡 TIP 3**: If you encounter a deprecated model error, ask CoCo *"Replace deprecated models in this notebook with current similar low-cost alternatives"* and it will fix them for you.

---

## Connect to a Service

Before running cells in this notebook, you must connect to a compute service.

**First time (create a new service):**
1. Click the **Connect** button at the top of this notebook
2. Click **Create Service** — a default name like `{{user}}_SERVICE1` will be suggested
3. Click **Service Settings** and select `ALLOW_ALL_EAI` as the external access integration
4. Leave other settings as default and click **Create**
5. Wait for the service to reach a **READY** state

**Returning (service already exists):**
1. Click the **Connect** button
2. Select your existing service from the list

Once connected, you can run Python and SQL cells interactively.


## Introduction

Snowflake is constantly working to improve the performance and cost-effectiveness of its platform. These ongoing efforts are aimed at ensuring that you, as a customer, gain maximum value from the workloads you run on Snowflake, including those involving AI-driven initiatives. Snowflake costs are based on the Credit Consumption table listed below. 



If this URL does not work copy and paste it from a new browser tab.

[Credit Consumption table](https://www.snowflake.com/legal-files/CreditConsumptionTable.pdf)

### Set up your current context for the role, database, schema and warehouse.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
your_db = user + '_GENAI_DB'
print('Your current CONTEXT information:')
print(session)

In [ ]:
%%sql -r Set_up_your_current_context_for_the_sql
USE ROLE genai_role;
USE DATABASE {{user}}_genai_db;
USE SCHEMA raw;
USE WAREHOUSE {{user}}_genai_wh;
ALTER SESSION SET query_tag = '{{user}} lab - TOPIC: Cost Management';
SHOW PARAMETERS LIKE 'query_tag' IN SESSION 
  ->> SELECT "value" AS query_tag FROM $1;

## Query Account Usage Views For Metering


### Observe the credit usage for AI Services for each day.

By executing the following query, we can view the credit usage for AI Services from the **SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY** view for each day.

Here's a breakdown of the key columns and what they represent:

**SERVICE_TYPE**:
This column confirms that the data pertains to **AI_SERVICES**, as specified in the query's WHERE clause.

**USAGE_DATE**:
This column indicates the date for which the credit usage is recorded.

**CREDITS_USED_COMPUTE**:
This column represents the number of Snowflake credits consumed by compute resources used for AI services.

**CREDITS_USED_CLOUD_SERVICES**: This column shows the credits used for Snowflake-managed cloud services that supported the AI operations.

**CREDITS_USED**:
This column represents the total number of Snowflake credits consumed by AI Services on the corresponding **USAGE_DATE**.

**CREDITS_ADJUSTMENTS_CLOUD_SERVICES**:
This column shows any adjustments, such as promotions or discounts, applied to the cloud services credits.

**CREDITS_BILLED**:
This column displays the billed credits for AI Services, which typically match **CREDITS_USED** for standard usage.

By analyzing the **CREDITS_USED** and **CREDITS_BILLED** columns, you can track the daily cost of using AI Services in your Snowflake account.

It is very important to understand that in many cases, especially when the AI service is the main driver of the cost, that the **CREDITS_USED** and the **CREDITS_BILLED** are the same number.

This view provides valuable insights into the consumption patterns of AI Services, allowing you to optimize your usage and manage costs effectively.
The **USAGE_DATE** allows you to track trends over time.

In [ ]:
%%sql -r Query_Account_Usage_metering_sql
USE ROLE genai_role;

SELECT * 
FROM  snowflake.account_usage.metering_daily_history
WHERE service_type='AI_SERVICES'
ORDER BY usage_date;


### Observe the hourly costs for AI Services. Which day/hour is most costly?
By executing the following query, we can view the **hourly** credits usage for AI Services from the **SNOWFLAKE.ACCOUNT_USAGE.METERING_HISTORY** view.

In [ ]:
%%sql -r Observe_the_hourly_costs_for_AI_sql
SELECT 
    start_time::TIMESTAMP AS hour_start_time,
    SUM(credits_used) AS total_credits_used,
    service_type
FROM 
    snowflake.account_usage.metering_history
WHERE service_type='AI_SERVICES'
GROUP BY 
    hour_start_time, service_type
ORDER BY 
    total_credits_used DESC;

### Check the functions usage and the tokens and credits associated with them.

Which functions are the most expensive?

In [ ]:
%%sql -r Check_the_functions_usage_and_the_sql
SELECT * 
FROM snowflake.account_usage.cortex_aisql_usage_history
ORDER BY token_credits DESC;

### Which Cortex function query is most expensive?

The following query retrieves details about the most expensive Cortex function query based on token usage. 
The query is made up of multiple steps.


**WHDATA CTE**:
Retrieves a distinct list of warehouses (warehouse_name, warehouse_id) from **WAREHOUSE_METERING_HISTORY**. This helps in mapping warehouse IDs to names for better readability.


**MAX_QUERY CTE**:
Fetches the query that used the highest number of tokens from **CORTEX_AISQL_USAGE_HISTORY** view.
Orders results by **TOKEN_CREDITS** DESC (highest to lowest). Uses LIMIT 1 to select only the most expensive query.


**MAIN QUERY**:
Fetches key details about the most expensive query, including:

**QUERY_ID**: Unique ID of the query

**QUERY_TEXT**: Actual query run

**FUNCTION_NAME**: Cortex function used

**TOKEN_CREDITS**: Tokens consumed

**WAREHOUSE_NAME**: Where the query ran

**START_TIME**: Start time

**END_TIME**: End time

**EXECUTION_STATUS**: Status of query e.g. SUCCESS

In [ ]:
%%sql -r Which_Cortex_function_query_is_most_sql
-- This query identifies the single most expensive Snowflake Cortex function call
-- based on the number of token credits used. It then retrieves detailed
-- information about that specific query, including its full text, the warehouse
-- it ran on, and its execution status.

-- Define a Common Table Expression (CTE) to get a unique list of warehouses and their IDs.
-- This is used later to translate the warehouse_id to a human-readable warehouse_name.
WITH whdata AS (
   SELECT DISTINCT
        warehouse_name,
        warehouse_id
    FROM
        snowflake.account_usage.warehouse_metering_history
),
-- Define a second CTE to find the single Cortex query with the highest token usage.
max_query AS (
    SELECT
        query_id,
        function_name,
        token_credits,
        warehouse_id
    FROM
        snowflake.account_usage.CORTEX_AISQL_USAGE_HISTORY
    ORDER BY
        token_credits DESC -- Order by token usage in descending order to find the most expensive.
    LIMIT 1                -- Restrict the result to only the top row (the most expensive query).
)
-- Final SELECT statement to assemble the detailed report.
SELECT
    max_query.query_id,        -- The unique identifier of the most expensive query.
    qhist.query_text,          -- The full SQL text of that query.
    max_query.function_name,   -- The name of the Cortex function that was called.
    max_query.token_credits,   -- The number of tokens consumed by the function call.
    whdata.warehouse_name,     -- The name of the virtual warehouse that ran the query.
    qhist.start_time,          -- Timestamp for when the query started.
    qhist.end_time,            -- Timestamp for when the query finished.
    qhist.execution_status     -- The final status of the query (e.g., SUCCESS, FAILED).
FROM max_query
-- Join with the general query history table to get the full query text and run times.
JOIN
    snowflake.account_usage.query_history AS qhist
    ON max_query.query_id = qhist.query_id
-- Join with the warehouse data CTE to get the warehouse name.
JOIN
    whdata
    ON max_query.warehouse_id = whdata.warehouse_id;



### View the Cost monitoring dashboard.

Snowflake provides a built-in dashboard for cost monitoring in Snowsight.

In Snowsight, under **GENAI_ROLE**, Click on **Admin**-> **Cost Management** -> Click on **Consumption**
Filter on **Service Type** - **AI Services**. Look for the line that says **AI Services**. You will see the credit usage for that service.

Review the costs for the Snowflake account. You will see total Credit used for a period that you have selected. i.e. for Last Day, Last 7 Days etc.

### View Cortex Search services frequency.

By running the following command we can see the search services in our snowflake account. 

In [ ]:
%%sql -r View_Cortex_Search_services_frequency_sql
USE  DATABASE {{user}}_genai_db;
USE SCHEMA resources;
USE ROLE genai_role;
SHOW CORTEX SEARCH SERVICES;

You can drill down into each search service and check what the lag is set to. Currently, it is set to 1 minute. In some instances, you may not have a real need to update the search index every minute. 

### Find out more details about the Cortex Search Services.

Using the query below, we can find out more details on the daily usage history of Cortex Search based on consumption category covering both serving and embedding. 

In [ ]:
%%sql -r Find_details_Cortex_Search_Services_sql
SELECT usage_date,service_name,model_name,tokens
FROM snowflake.account_usage.cortex_search_daily_usage_history
ORDER BY CREDITS DESC
LIMIT 5;


### Find out more details on credits usage for Cortex Analyst.

The following query retrieves data from the **SNOWFLAKE.ACCOUNT_USAGE.CORTEX_ANALYST_USAGE_HISTORY** view and filters for the top 5 records with the highest request count.

**START_TIME and END_TIME**: These columns, both of the TIMESTAMP_LTZ data type, indicate the beginning and end of the one-hour window in which Cortex Analyst messages were processed.

**REQUEST_COUNT**: This NUMBER column shows the total count of messages sent to Cortex Analyst within that time frame.

**CREDITS**: A NUMBER column that reflects the quantity of credits billed for the Cortex Analyst messages processed.

**USERNAME**: This TEXT column identifies the user who initiated the Cortex Analyst message request.

In essence, this view allows you to monitor the cost and usage of Cortex Analyst by tracking credit consumption, message volume, and user activity over time. The data is aggregated in one-hour increments and is available for the past year.

In [ ]:
%%sql -r Find_details_credits_Cortex_Analyst_sql
SELECT *
FROM snowflake.account_usage.cortex_analyst_usage_history
ORDER BY REQUEST_COUNT DESC
LIMIT 5;

### Find out more details on credits usage for Cortex Search.


The view queried below allows you to monitor and analyze the costs associated with running your Cortex Search Services. Specifically, it tracks the credits used for "serving," which is the operational cost of keeping your search service active and responsive to queries. It's important to note that this view does not include other costs, such as those related to embedding text.

The **CORTEX_SEARCH_SERVING_USAGE_HISTORY** view in Snowflake is designed to provide a detailed, hourly breakdown of the credits consumed by the "serving" aspect of your Cortex Search Services. 

**START_TIME and END_TIME**: These TIMESTAMP_LTZ columns define the one-hour window during which the usage data was recorded.

**DATABASE_NAME and SCHEMA_NAME**: These TEXT columns specify the location of your Cortex Search Service within your Snowflake environment.

**SERVICE_NAME and SERVICE_ID**: These columns, TEXT and NUMBER respectively, identify the specific Cortex Search Service for which the usage is being reported.

**CREDITS**: This NUMBER column shows the total number of credits that were billed for the serving usage of the specified service during the recorded time window.

By querying this view, you can gain a granular understanding of how your Cortex Search Services are consuming credits, which is crucial for cost management and optimization. The data in this view is retained for a full year (365 days).


The query below retrieves data from the **SNOWFLAKE.ACCOUNT_USAGE.CORTEX_SEARCH_SERVING_USAGE_HISTORY** view and filters for the top 5 records with the highest credits usage.




In [ ]:
%%sql -r Find_details_credits_Cortex_Search_sql
SELECT 
    start_time,
    end_time,
    database_name,
    schema_name,
    service_name,
    service_id,
    credits
FROM snowflake.account_usage.cortex_search_serving_usage_history
ORDER BY credits DESC
LIMIT 5;  


In [ ]:
%%sql -r Find_details_credits_Cortex_Search_describe_sql
DESCRIBE CORTEX SEARCH SERVICE travelbug_search;

### Suspend search services.

If you have reached the end of the lab, you can suspend the search services indexing.

In [ ]:
%%sql -r Suspend_search_services_sql
ALTER CORTEX SEARCH SERVICE  IF EXISTS travelbug_search  SUSPEND INDEXING;

In [ ]:
%%sql -r Suspend_search_services_structured_sql
ALTER CORTEX SEARCH SERVICE  IF EXISTS  travelbug_search_structured  SUSPEND INDEXING;

In [ ]:
%%sql -r Suspend_search_services_structured_chunks_sql
ALTER CORTEX SEARCH SERVICE IF EXISTS  travelbug_search_structured_chunks SUSPEND INDEXING;

### Find out more details on document processing usage.

The CORTEX_DOCUMENT_PROCESSING_USAGE_HISTORY view in Snowflake is a valuable tool for monitoring and understanding your usage of document processing features. Here's a summary of its key aspects:

This view provides a detailed, hourly aggregation of your document processing activities. It allows you to track which functions and models are being used, how many pages are being processed, and, most importantly, the number of credits consumed. This is essential for cost management and for optimizing your document processing workflows.


**QUERY_ID**: A VARCHAR that uniquely identifies the SQL query that initiated the document processing job.

**CREDITS_USED**: A NUMBER indicating the total credits billed for the document processing function in that specific query.

**START_TIME and END_TIME**: These TIMESTAMP_LTZ columns show the start and end times of the query.

**FUNCTION_NAME**: A TEXT column that specifies the name of the document processing function used (e.g. AI_PARSE_DOCUMENT, AI_EXTRACT).

**MODEL_NAME**: A TEXT column that identifies the model used for the processing task.

**PAGE_COUNT**: A NUMBER representing the total number of pages processed.

**DOCUMENT_COUNT**: A NUMBER indicating the total number of documents processed.

**FEATURE_COUNT**: A NUMBER that reflects the number of data values defined for extraction in the document processing operation.

By using this view, you can gain a clear understanding of your document processing usage and costs. The data in this view is retained for a full year (365 days).

In [ ]:
%%sql -r Find_out_more_details_on_document_sql
SELECT * 
FROM snowflake.account_usage.cortex_document_processing_usage_history
ORDER BY CREDITS_USED DESC
LIMIT 5;

### Find out more details on credits usage for Cortex Fine-Tuning.

The query retrieves data from the **SNOWFLAKE.ACCOUNT_USAGE.CORTEX_FINE_TUNING_USAGE_HISTORY** view and filters for the top 5 records with the highest credits usage.

In [ ]:
%%sql -r Find_details_credits_Cortex_Fine_Tuning_sql
SELECT * 
FROM snowflake.account_usage.cortex_fine_tuning_usage_history
ORDER BY token_credits DESC
LIMIT 5;

### Setting up Resource Budgets for Cortex Agents and Snowflake CoWork.

Resource budgets let you set a monthly credit spending limit on AI services and take automated actions (like notifications or access revocation) when spending exceeds thresholds. They use Snowflake's **tag-based cost attribution** model and the `SNOWFLAKE.CORE.BUDGET` class.

**Supported AI services:**
| Service | Tag applied to | Domain in `ADD_SHARED_RESOURCE` |
|---------|---------------|----------------------------------|
| Cortex Agents | The Agent object | `'CORTEX AGENT'` |
| Snowflake CoWork | The CoWork (SI) object | `'SNOWFLAKE INTELLIGENCE'` |

**How it works (same pattern for both):**

1. **Create a tag** and apply it to the object (Agent or CoWork instance).
2. **Create a budget** (`SNOWFLAKE.CORE.BUDGET`) and set a monthly spending limit.
3. **Associate the tag** with the budget using `SET_RESOURCE_TAGS`.
4. **Configure threshold actions**, which are stored procedures that fire at spending thresholds (e.g., notify at 80%, revoke access at 100%).
5. **Monitor usage** by calling `GET_SERVICE_TYPE_USAGE_V2` to view credits consumed.

**Enforcement latency:** Up to 8 hours by default (2 hours with low-latency budget enabled). Set alerts at 80% to give yourself time to respond before a 100% hard stop.

**Shared resource budgets (per-team):** For teams sharing the same Agent or CoWork, you can create shared resource budgets that tag users instead of objects, allowing independent limits per team.  

Refer to:
- [Resource budgets for Cortex Agents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-resource-budgets)
- [Resource budgets for Snowflake CoWork](https://docs.snowflake.com/en/user-guide/snowflake-cortex/snowflake-cowork/cowork-resource-budgets)

In this lab we will not be setting up resource budgets. 


### Cost management guidelines.

It is crucial to monitor costs for the following through Snowsight dashboards, custom queries, and other features such as budgets and resource monitors to prevent cost overruns.

* Managing Costs for LLM Functions:
Snowflake recommends using a warehouse size no larger than MEDIUM when calling Snowflake Cortex LLM functions. Using a larger warehouse than necessary does not increase performance but can result in unnecessary costs and a higher risk of throttling.

* Managing Cross-Region Costs:
Understand how cross-region inference works before enabling it. You are charged credits based on the region where the request originates, not where it is processed. For example, if a request is made from the us-east-2 region and processed in us-west-2, credits are consumed in us-east-2. No Data Egress Charges: There are no additional data egress fees for cross-region inference.

* Managing Document Processing Costs:
For document content extraction, several factors need to be considered for costs, including the number of pages, number of documents, page density, and data values to extract.

* Managing Costs for Cortex Search:
Use a warehouse no larger than MEDIUM. Additional costs apply for AI services serving costs, embedding costs, cloud services costs, and storage costs.

* Managing Cortex Agents Costs:
Use resource budgets to monitor and control Cortex Agents spend. Apply a tag to your Agent object, associate it with a budget, and configure threshold actions - such as notifications at 80% and access revocation at 100% of the monthly credit limit. See [Resource budgets for Cortex Agents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-resource-budgets).

* Managing Snowflake CoCo Costs:
Account administrators can set daily estimated credit usage limits for Snowflake CoCo on a per-user basis using the **CORTEX_CODE_CLI_DAILY_EST_CREDIT_LIMIT_PER_USER** and **CORTEX_CODE_SNOWSIGHT_DAILY_EST_CREDIT_LIMIT_PER_USER** parameters. These limits track usage over a rolling 24-hour window and block access when exceeded. Limits can be set at the account level or per user. See [Cost controls for Snowflake CoCo](https://docs.snowflake.com/en/user-guide/cortex-code/credit-usage-limit).

* Best Practices:
Start with monitoring before implementing automated controls. Use **QUERY_TAG** session parameters to enable cost attribution by project or team. Set conservative initial limits and adjust based on actual usage. The ACCOUNT_USAGE views have up to 60 minutes of latency - factor this into your monitoring strategy. See [Managing Cortex AI Function costs](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-func-cost-management).

Refer to the snowflake documentation for costs for each of these features.

* https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst
* https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview
* https://docs.snowflake.com/en/user-guide/snowflake-cortex/parse-document
* https://docs.snowflake.com/en/user-guide/snowflake-cortex/cross-region-inference
* https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-func-cost-management
* https://docs.snowflake.com/en/user-guide/cortex-code/credit-usage-limit
* https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-resource-budgets
* https://docs.snowflake.com/en/user-guide/snowflake-cortex/snowflake-cowork/cowork-resource-budgets

## 🎯 Challenge Questions

Test your understanding of the concepts covered in this lab.

In [ ]:
from snowflake.snowpark.context import get_active_session
from IPython.display import display, HTML

session = get_active_session()

quiz_data = [
    {"q": "Which Snowflake views can you use to track Cortex AI service costs?", "options": ["A) Only QUERY_HISTORY view", "B) METERING_DAILY_HISTORY and related views track Cortex AI service consumption", "C) Only external billing systems", "D) Cost data is not available in Snowflake"], "hash": "d427de5d11400b98e6790eca3bc3f1ef"},
    {"q": "How does token usage affect Cortex costs?", "options": ["A) Tokens have no impact on cost", "B) Only input tokens are charged", "C) Token usage directly impacts the cost of Cortex LLM function calls", "D) Token limits are only for performance"], "hash": "ae2431ff4ef31260afbf90ee8e4bdd19"},
    {"q": "What happens when you suspend a Cortex Search service?", "options": ["A) Suspending Cortex Search services stops ongoing credit consumption", "B) The service data is permanently deleted", "C) Queries continue to work but slower", "D) Credits continue to be consumed"], "hash": "4f6a11ee09e1bc6aaf7e8fc809e0fdf3"},
    {"q": "Which view provides the most detailed Cortex function usage information?", "options": ["A) WAREHOUSE_METERING_HISTORY only", "B) QUERY_HISTORY only", "C) LOGIN_HISTORY view", "D) The CORTEX_AISQL_USAGE_HISTORY view provides detailed function-level cost data"], "hash": "3b5155a5aa4e0c0c986e50deb7f1254f"},
    {"q": "Why is cost monitoring important for Cortex AI workloads?", "options": ["A) Cost monitoring is only required for compliance", "B) Monitoring costs helps optimize AI workloads and control spending on Cortex services", "C) Costs are fixed regardless of usage", "D) Cost monitoring is optional and has no practical benefit"], "hash": "2702ffc2d46536cd3299708b7913f78e"},
]

results_map = {}
for qi, item in enumerate(quiz_data):
    results_map[qi] = {}
    for opt in item["options"]:
        letter = opt[0]
        escaped_opt = opt.replace("'", "''")
        result = session.sql(f"CALL genai_db.resources.quiz_temp('{item['hash']}', '{escaped_opt}', 'False')").collect()
        feedback = result[0][0]
        is_correct = 'Correct' in feedback or '✅' in feedback
        results_map[qi][letter] = (feedback, is_correct)

html = """<style>
.cq { margin: 20px 0; padding: 16px; border: 1px solid #d0d0d0; border-radius: 10px; background: #fafafa; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif; font-size: 14px; }
.cq h4 { font-family: inherit; }
.cq input[type="radio"] { display: none; }
.cq .lbl { display: block; padding: 8px 12px; border-radius: 6px; cursor: pointer; font-family: inherit; }
.cq .lbl:hover { background: #e8f0fe; }
.cq input[type="radio"]:checked + .lbl { border-color: #1a73e8; background: #e8f0fe; font-weight: 600; }
.cq .fb { display: none; padding: 6px 12px; margin-top: 2px; border-radius: 4px; font-weight: 600; font-family: inherit; }
.cq input[type="radio"]:checked + .lbl + .fb { display: block; }
.cq .fb.ok { background: #e6f4ea; color: #1e7e34; }
.cq .fb.no { background: #fce8e6; color: #c62828; }
</style>"""

for qi, item in enumerate(quiz_data):
    html += f'<div class="cq"><h4>Q{qi+1}: {item["q"]}</h4>'
    for opt in item["options"]:
        letter = opt[0]
        feedback, is_correct = results_map[qi][letter]
        css_class = 'ok' if is_correct else 'no'
        uid = f'cq{qi}_{letter}'
        html += f'<div class="opt"><input type="radio" name="cq{qi}" id="{uid}">'
        html += f'<label class="lbl" for="{uid}">{opt}</label>'
        html += f'<div class="fb {css_class}">{letter}) {feedback}</div></div>'
    html += '</div>'

display(HTML(html))

## Key Takeaways

❄️ Costs can be monitored through SNOWFLAKE.ACCOUNT_USAGE views and Cost Management dashboards.

❄️ Resource budgets (`SNOWFLAKE.CORE.BUDGET`) let you set monthly credit limits on Cortex Agents and Snowflake CoWork, with automated notifications and access revocation at configurable thresholds.

❄️ Understand cost management guidelines by referring to the documentation.